In [ ]:
import numpy as np
import glob
import matplotlib.pyplot as pl
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
def correlation_from_covariance(covariance):
    v = np.sqrt(np.diag(covariance))
    outer_v = np.outer(v, v)
    correlation = covariance / outer_v
    correlation[covariance == 0] = 0
    return correlation

In [ ]:
# From Mike's files
pkpath = '/Users/austerlitz/TripoSH-Factory/my_project/data/2ndGen/QSO/latest'
bkpath = '/Users/austerlitz/TripoSH-Factory/my_project/data/2ndGen/QSO/latest'

pk0 = np.load(pkpath+'/pk0_cubicbox_6gpc_QSO_z1.400_mocks.npy',allow_pickle=True).item()
pk2 = np.load(pkpath+'/pk2_cubicbox_6gpc_QSO_z1.400_mocks.npy',allow_pickle=True).item()
pk4 = np.load(pkpath+'/pk4_cubicbox_6gpc_QSO_z1.400_mocks.npy',allow_pickle=True).item()
bk000 = np.load(bkpath+'/bk000_diag_cubicbox_6gpc_QSO_z1.400_mocks.npy',allow_pickle=True).item()
bk202 = np.load(bkpath+'/bk202_diag_cubicbox_6gpc_QSO_z1.400_mocks.npy',allow_pickle=True).item()

In [ ]:
Nmocks = bk202['stats'].shape[0]

stats = [np.concatenate((pk0['stats'][i,:], 
                      pk2['stats'][i,:], 
                      pk4['stats'][i,:],
                      bk000['stats'][i,:],
                      bk202['stats'][i,:])) for i in range(0,Nmocks)]

stats = np.stack(stats)
cov = np.cov(stats.T)

In [ ]:
pl.imshow(correlation_from_covariance(cov))
pl.colorbar()

In [ ]:
# Number of mocks
N_mocks   = Nmocks
# Number of wavemodes (so far, pk and bk agree...)
nk        = len(pk0['coords'])
# Number of summary statistics to be considered: multipoles = 0,2,4,000,202
nstats    = 5

In [ ]:
vol_ratio = 6**3/2**3

k_Pk_0 = pk0['coords']
k_Pk_2 = pk2['coords']
k_Pk_4 = pk4['coords']
k_Bk_000 = bk000['coords']
k_Bk_202 = bk202['coords']

length_multi = []
for i in range(nstats):
    length_multi.append(nk)

# Construct the covariance dictionary for the new fitting pipeline
result_cova = {
    'nmocks': N_mocks,
    'k': { '0': k_Pk_0,
           '2': k_Pk_2,
           '4': k_Pk_4,
           '000': k_Bk_000,
           '202': k_Bk_202
         },
    'length_multi': length_multi,
    'cov': cov*vol_ratio
}

np.save('/data/QSO/cov_QSO_0-2-4_000-202.npy',result_cova)